# Pipeline de Validación e Ingesta → Base de Datos FIA
**Archivo fuente:** `Dep FIA climas riegos sondas.xlsx`
**Tabla destino:** `MedicionesClimaticas`
**Hoja:** `MedicionesClimaticas` | 1.860 filas × 15 columnas

---
### Flujo
```
Excel → Carga y renombrado → Validación → Limpieza → Vista previa → Ingesta SQL
```


## 0 · Conexión a la base de datos FIA

In [4]:
import pyodbc
import pandas as pd
import numpy as np
from datetime import datetime

# ── Conexión (mismo string que ya funciona) ────────────────────────────────────
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=LAPTOP-PUELCHE\\SQLEXPRESS;"
    "DATABASE=CH_UCHILE;"
    "UID=sa;"
    "PWD= <Personal_key>;"
    "TrustServerCertificate=yes;"
)

try:
    cnxn   = pyodbc.connect(conn_str)
    cursor = cnxn.cursor()
    print("✅ Conexión exitosa a FIA")
except pyodbc.Error as e:
    print(f"❌ Error de conexión: {e}")

❌ Error de conexión: ('28000', "[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user 'sa'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user 'sa'. (18456)")


## 1 · Carga del archivo Excel y estandarización de nombres
El renombrado se hace **inmediatamente después de cargar** el Excel (y no en la sección de Limpieza como antes), para que las reglas de validación de la sección 2 ya trabajen con los nombres definitivos.

In [5]:
# ── Ruta del archivo (ajusta si es necesario) ──────────────────────────────────
EXCEL_PATH  = r"C:/Users/marce/OneDrive/Documentos/OneDrive/Documentos/Chirimoyos_FIA/Terecer_envió_info_20260529/Dep FIA climas riegos sondas.xlsx"
SHEET_NAME  = "MedicionesClimaticas"
TABLE_DEST  = "MedicionesClimaticas"        # nombre de la tabla en SQL Server

# ── Diccionario de renombrado: encabezado Excel → nombre SQL-friendly ─────────
RENOMBRE_COLUMNAS = {
    "Fecha"                                : "fecha",
    "Tratamiento"                          : "tratamiento",
    "Temperatura minima (ºC)"              : "temp_min",
    "Temperatura media (ºC)"               : "temp_media",
    "Temperatura maxima (ºC)"              : "temp_max",
    "Humedad relativa minima (%)"          : "hr_min",
    "Humedad relativa media (%)"           : "hr_media",
    "Humedad relativa maxima (%)"          : "hr_max",
    "Precipitaciones (mm)"                 : "precipitaciones_mm",
    "Piranómetro RG (W/m2)"                : "piranometro_rg",        # ← variable nueva
    "Rad diaria (MJ/m2*dia)"               : "rad_diaria_mj",         # ← variable nueva
    "Integral luz diaria ILD (mol/m2*día)" : "ild_mol",               # ← variable nueva
    "DPV (Kpa)"                            : "dpv_kpa",
    "Horas Frío"                           : "horas_frio",
    "HF acumuladas"                        : "hf_acumuladas",
}

# ── Carga ─────────────────────────────────────────────────────────────────────
df_raw = pd.read_excel(
    EXCEL_PATH,
    sheet_name = SHEET_NAME,
)

# ── Renombrado inmediato ───────────────────────────────────────────────────────
df_raw.rename(columns=RENOMBRE_COLUMNAS, inplace=True)

# ── Chequeo de columnas esperadas (avisa si el Excel trae encabezados distintos)
faltantes = set(RENOMBRE_COLUMNAS.values()) - set(df_raw.columns)
if faltantes:
    print(f"⚠️  Columnas esperadas que no llegaron desde el Excel: {faltantes}")

print(f"Filas cargadas : {len(df_raw):>6,}")
print(f"Columnas       : {len(df_raw.columns):>6}")
print(f"Período        : {df_raw['fecha'].min().date()}  →  {df_raw['fecha'].max().date()}")
print()
df_raw.head(15)

Filas cargadas :  1,860
Columnas       :     15
Período        : 2023-11-11  →  2026-05-28



,fecha,tratamiento,temp_min,temp_media,temp_max,hr_min,hr_media,hr_max,precipitaciones_mm,piranometro_rg,rad_diaria_mj,ild_mol,dpv_kpa,horas_frio,hf_acumuladas
0,2023-11-11,Campo,10.5,16.1,21.7,58.3,78.8,99.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-11-12,Campo,7.9,14.8,21.7,52.5,73.4,94.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-11-13,Campo,7.6,15.0,22.4,46.2,68.7,91.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-11-14,Campo,9.3,14.2,19.0,60.2,78.1,94.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-11-15,Campo,8.9,14.3,19.7,53.9,75.4,94.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-11-16,Campo,6.6,14.0,21.3,46.9,70.4,91.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2023-11-17,Campo,7.8,13.8,19.9,53.2,74.0,93.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2023-11-18,Campo,7.6,14.6,21.7,49.5,72.7,93.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2023-11-19,Campo,9.3,14.2,19.2,64.6,83.4,98.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2023-11-20,Campo,11.8,15.4,19.1,61.8,77.9,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2 · Validación de datos
Ejecuta 8 reglas de negocio sobre el dataset crudo (ya renombrado) y genera un **reporte de calidad**. Se agregó la Regla 8 para las nuevas variables de radiación/luz.

In [6]:
errores = []   # lista de dicts con cada problema encontrado

# ── Regla 1: Nulos en columnas obligatorias ────────────────────────────────────
OBLIGATORIAS = ["fecha", "tratamiento", "temp_min", "temp_media", "temp_max",
                "hr_min", "hr_media", "hr_max"]
for col in OBLIGATORIAS:
    n = df_raw[col].isnull().sum()
    if n > 0:
        errores.append({"regla": "Nulo_obligatorio", "columna": col,
                        "n_filas": n, "indices": list(df_raw[df_raw[col].isnull()].index[:5])})

# ── Regla 2: Duplicados (fecha + tratamiento) ──────────────────────────────────
mask_dup = df_raw.duplicated(subset=["fecha", "tratamiento"], keep=False)
n_dup = mask_dup.sum()
if n_dup > 0:
    errores.append({"regla": "Duplicado", "columna": "fecha+tratamiento",
                    "n_filas": n_dup, "indices": list(df_raw[mask_dup].index[:5])})

# ── Regla 3: Coherencia temperatura (min ≤ media ≤ max) ───────────────────────
mask_t = (df_raw["temp_min"] > df_raw["temp_media"]) | (df_raw["temp_media"] > df_raw["temp_max"])
if mask_t.sum() > 0:
    errores.append({"regla": "Coherencia_temp", "columna": "temp_min/media/max",
                    "n_filas": int(mask_t.sum()), "indices": list(df_raw[mask_t].index[:5])})

# ── Regla 4: Coherencia humedad relativa (min ≤ media ≤ max) ──────────────────
mask_h = (df_raw["hr_min"] > df_raw["hr_media"]) | (df_raw["hr_media"] > df_raw["hr_max"])
if mask_h.sum() > 0:
    errores.append({"regla": "Coherencia_HR", "columna": "hr_min/media/max",
                    "n_filas": int(mask_h.sum()), "indices": list(df_raw[mask_h].index[:5])})

# ── Regla 5: Rango válido temperatura ─────────────────────────────────────────
# Agronomía: chirimoyos → rango realista -5°C a 45°C
mask_tr = (df_raw["temp_min"] < -5) | (df_raw["temp_max"] > 45)
if mask_tr.sum() > 0:
    errores.append({"regla": "Outlier_temp", "columna": "temp_min/temp_max",
                    "n_filas": int(mask_tr.sum()), "indices": list(df_raw[mask_tr].index[:5])})

# ── Regla 6: Rango válido HR (0–100%) ─────────────────────────────────────────
mask_hr = (df_raw["hr_min"] < 0) | (df_raw["hr_max"] > 100)
if mask_hr.sum() > 0:
    errores.append({"regla": "Outlier_HR", "columna": "hr_min/hr_max",
                    "n_filas": int(mask_hr.sum()), "indices": list(df_raw[mask_hr].index[:5])})

# ── Regla 7: Precipitaciones no negativas ─────────────────────────────────────
mask_pp = df_raw["precipitaciones_mm"].notna() & (df_raw["precipitaciones_mm"] < 0)
if mask_pp.sum() > 0:
    errores.append({"regla": "Negativo_precip", "columna": "precipitaciones_mm",
                    "n_filas": int(mask_pp.sum()), "indices": list(df_raw[mask_pp].index[:5])})

# ── Regla 8: Variables de radiación/luz no negativas (variables nuevas) ──────
COLS_RADIACION = ["piranometro_rg", "rad_diaria_mj", "ild_mol"]
for col in COLS_RADIACION:
    mask_rad = df_raw[col].notna() & (df_raw[col] < 0)
    if mask_rad.sum() > 0:
        errores.append({"regla": "Negativo_radiacion", "columna": col,
                        "n_filas": int(mask_rad.sum()), "indices": list(df_raw[mask_rad].index[:5])})

# ── Reporte ────────────────────────────────────────────────────────────────────
print("=" * 65)
print(" REPORTE DE CALIDAD")
print("=" * 65)
print(f" Total filas analizadas : {len(df_raw):,}")
print(f" Problemas detectados   : {len(errores)}")
print("-" * 65)

if errores:
    df_errores = pd.DataFrame(errores)
    print(df_errores.to_string(index=False))
    print()
    print("⚠️  Revisa la celda de LIMPIEZA antes de ingestar.")
else:
    print(" ✅ Sin errores críticos. Dataset listo para ingesta.")
print("=" * 65)

 REPORTE DE CALIDAD
 Total filas analizadas : 1,860
 Problemas detectados   : 1
-----------------------------------------------------------------
       regla           columna  n_filas indices
Outlier_temp temp_min/temp_max        1   [889]

⚠️  Revisa la celda de LIMPIEZA antes de ingestar.


## 3 · Limpieza y transformación
Corrige los problemas detectados y agrega columnas de auditoría. (El renombrado ya se hizo en la sección 1, así que aquí solo quedan las transformaciones adicionales.)

In [9]:
df = df_raw.copy()

# ── 3.1  Estandarizar texto en tratamiento ─────────────────────────────────────
df["tratamiento"] = df["tratamiento"].str.strip().str.capitalize()

# ── 3.2  Marcar filas outlier temperatura (temp_max > 45) como flag ───────────
#         NO se eliminan; se marcan para auditoría
df["flag_revision"] = False
mask_outlier_t = (df["temp_min"] < -5) | (df["temp_max"] > 45)
df.loc[mask_outlier_t, "flag_revision"] = True

# ── 3.3  Agregar columna de carga (trazabilidad) ───────────────────────────────
df["fecha_carga"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ── 3.4  Resumen post-limpieza ─────────────────────────────────────────────────
print(f"Filas listas para ingestar : {len(df):,}")
print(f"Filas con flag_revision    : {df['flag_revision'].sum()}")
print()
print("Columnas finales:")
for col, dtype in df.dtypes.items():
    nulos = df[col].isnull().sum()
    print(f"  {col:<25} {str(dtype):<15} nulos: {nulos}")
print()
df.head(3)

Filas listas para ingestar : 1,860
Filas con flag_revision    : 1

Columnas finales:
  fecha                     datetime64[ns]  nulos: 0
  tratamiento               object          nulos: 0
  temp_min                  float64         nulos: 0
  temp_media                float64         nulos: 0
  temp_max                  float64         nulos: 0
  hr_min                    float64         nulos: 0
  hr_media                  float64         nulos: 0
  hr_max                    float64         nulos: 0
  precipitaciones_mm        float64         nulos: 1012
  piranometro_rg            float64         nulos: 164
  rad_diaria_mj             float64         nulos: 164
  ild_mol                   float64         nulos: 164
  dpv_kpa                   float64         nulos: 164
  horas_frio                float64         nulos: 82
  hf_acumuladas             float64         nulos: 164
  flag_revision             bool            nulos: 0
  fecha_carga               object          nulos: 0


,fecha,tratamiento,temp_min,temp_media,temp_max,hr_min,hr_media,hr_max,precipitaciones_mm,piranometro_rg,rad_diaria_mj,ild_mol,dpv_kpa,horas_frio,hf_acumuladas,flag_revision,fecha_carga
0,2023-11-11,Campo,10.5,16.1,21.7,58.3,78.8,99.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,2026-07-09 18:31:38
1,2023-11-12,Campo,7.9,14.8,21.7,52.5,73.4,94.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,2026-07-09 18:31:38
2,2023-11-13,Campo,7.6,15.0,22.4,46.2,68.7,91.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,2026-07-09 18:31:38


## 4 · Crear tabla en SQL Server (solo primera vez)

In [12]:
# ── DDL de la tabla destino ────────────────────────────────────────────────────
# Ejecuta esta celda SOLO la primera vez para crear la tabla.
# Si la tabla ya existe, esta versión no la vuelve a crear (usa IF NOT EXISTS).
# Nota: si ya tienes la tabla creada con la estructura ANTERIOR (sin las 3 columnas
# nuevas), tendrás que agregarlas manualmente con ALTER TABLE o borrar y recrear la tabla.

ddl = f"""
IF NOT EXISTS (
    SELECT 1 FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_NAME = '{TABLE_DEST}'
)
BEGIN
    CREATE TABLE {TABLE_DEST} (
        id                 INT IDENTITY(1,1) PRIMARY KEY,
        fecha              DATE          NOT NULL,
        tratamiento        NVARCHAR(50)  NOT NULL,
        temp_min           FLOAT,
        temp_media         FLOAT,
        temp_max           FLOAT,
        hr_min             FLOAT,
        hr_media           FLOAT,
        hr_max             FLOAT,
        precipitaciones_mm FLOAT,
        piranometro_rg     FLOAT,
        rad_diaria_mj      FLOAT,
        ild_mol            FLOAT,
        dpv_kpa            FLOAT,
        horas_frio         FLOAT,
        hf_acumuladas      FLOAT,
        flag_revision      BIT           NOT NULL DEFAULT 0,
        fecha_carga        DATETIME      NOT NULL
    );
    PRINT 'Tabla {TABLE_DEST} creada correctamente.';
END
ELSE
BEGIN
    -- Si la tabla ya existía sin las columnas nuevas, las agrega (no falla si ya están)
    IF NOT EXISTS (SELECT 1 FROM sys.columns WHERE Name = 'piranometro_rg' AND Object_ID = OBJECT_ID('{TABLE_DEST}'))
        ALTER TABLE {TABLE_DEST} ADD piranometro_rg FLOAT;
    IF NOT EXISTS (SELECT 1 FROM sys.columns WHERE Name = 'rad_diaria_mj' AND Object_ID = OBJECT_ID('{TABLE_DEST}'))
        ALTER TABLE {TABLE_DEST} ADD rad_diaria_mj FLOAT;
    IF NOT EXISTS (SELECT 1 FROM sys.columns WHERE Name = 'ild_mol' AND Object_ID = OBJECT_ID('{TABLE_DEST}'))
        ALTER TABLE {TABLE_DEST} ADD ild_mol FLOAT;
    PRINT 'Tabla {TABLE_DEST} ya existe (columnas nuevas verificadas/agregadas).';
END
"""

cursor.execute(ddl)
cnxn.commit()
print("✅ DDL ejecutado")

NameError: name 'cursor' is not defined

## 5 · Ingesta a SQL Server
Inserta los datos en lotes (`BATCH_SIZE`) con barra de progreso y manejo de errores por fila.

**`MODO_CARGA`**: como el Excel trae el histórico completo cada vez (no solo filas nuevas), volver a correr el notebook sin vaciar la tabla genera duplicados — esto es lo que pasó en la corrida anterior (13.020 filas en SQL vs. 1.860 en el Excel). Por eso, por defecto, se vacía la tabla antes de insertar (`"reemplazar"`). Si en el futuro el Excel solo trae filas nuevas, cambia a `"incremental"`.

In [13]:
import math

BATCH_SIZE  = 200          # filas por lote (ajustable)
MODO_CARGA  = "reemplazar" # "reemplazar" = vacía la tabla y recarga todo el histórico (recomendado)
                            # "incremental" = inserta sin borrar nada

if MODO_CARGA == "reemplazar":
    cursor.execute(f"DELETE FROM {TABLE_DEST}")
    cnxn.commit()
    print(f"🗑️  Tabla {TABLE_DEST} vaciada antes de la carga (modo reemplazar).")

# ── Preparar columnas en el orden exacto del DDL ──────────────────────────────
COLS_SQL = [
    "fecha", "tratamiento",
    "temp_min", "temp_media", "temp_max",
    "hr_min", "hr_media", "hr_max",
    "precipitaciones_mm", "piranometro_rg", "rad_diaria_mj", "ild_mol",
    "dpv_kpa", "horas_frio", "hf_acumuladas",
    "flag_revision", "fecha_carga"
]

placeholders = ", ".join(["?"] * len(COLS_SQL))
cols_str     = ", ".join(COLS_SQL)
sql_insert   = f"INSERT INTO {TABLE_DEST} ({cols_str}) VALUES ({placeholders})"

# ── Convertir NaN → None (pyodbc los pasa como NULL) ─────────────────────────
def fila_a_tupla(row):
    return tuple(None if (isinstance(v, float) and math.isnan(v)) else v
                 for v in row[COLS_SQL])

# ── Ingesta por lotes ─────────────────────────────────────────────────────────
total      = len(df)
insertadas = 0
fallos     = []
n_lotes    = math.ceil(total / BATCH_SIZE)

print(f"Iniciando ingesta: {total:,} filas en {n_lotes} lotes de {BATCH_SIZE}")
print("-" * 50)

for lote_i in range(n_lotes):
    inicio = lote_i * BATCH_SIZE
    fin    = min(inicio + BATCH_SIZE, total)
    lote   = df.iloc[inicio:fin]

    try:
        datos = [fila_a_tupla(row) for _, row in lote.iterrows()]
        cursor.executemany(sql_insert, datos)
        cnxn.commit()
        insertadas += len(lote)
        pct = insertadas / total * 100
        barra = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"\r  [{barra}] {pct:5.1f}%  ({insertadas:,}/{total:,})", end="", flush=True)

    except pyodbc.Error as e:
        cnxn.rollback()
        fallos.append({"lote": lote_i, "filas": f"{inicio}-{fin}", "error": str(e)[:120]})

print()
print("-" * 50)
print(f"✅ Insertadas : {insertadas:,}")
print(f"❌ Lotes fallidos : {len(fallos)}")
if fallos:
    print(pd.DataFrame(fallos).to_string(index=False))

NameError: name 'cursor' is not defined

## 6 · Verificación post-ingesta

In [ ]:
# ── Conteo total en la tabla ───────────────────────────────────────────────────
cursor.execute(f"SELECT COUNT(*) FROM {TABLE_DEST}")
total_sql = cursor.fetchone()[0]

print(f"Filas en Excel          : {len(df):>6,}")
print(f"Filas en SQL Server     : {total_sql:>6,}")
print(f"Coincidencia            : {'✅ OK' if total_sql == len(df) else '⚠️  Revisar'}")
print()

# ── Distribución por tratamiento (incluye promedios de las variables nuevas) ──
q_trat = f"""
    SELECT tratamiento,
           COUNT(*)                    AS n_registros,
           MIN(fecha)                  AS fecha_inicio,
           MAX(fecha)                  AS fecha_fin,
           ROUND(AVG(temp_media), 2)   AS temp_media_prom,
           ROUND(AVG(piranometro_rg), 1) AS piranometro_prom,
           ROUND(AVG(rad_diaria_mj), 2)  AS rad_diaria_prom,
           ROUND(AVG(ild_mol), 2)        AS ild_prom
    FROM {TABLE_DEST}
    GROUP BY tratamiento
    ORDER BY tratamiento
"""
df_check = pd.read_sql(q_trat, cnxn)
print("Distribución por tratamiento:")
print(df_check.to_string(index=False))
print()

# ── Filas marcadas para revisión ──────────────────────────────────────────────
cursor.execute(f"SELECT COUNT(*) FROM {TABLE_DEST} WHERE flag_revision = 1")
n_flags = cursor.fetchone()[0]
print(f"Filas con flag_revision=1 : {n_flags}")
if n_flags > 0:
    df_flags = pd.read_sql(
        f"SELECT fecha, tratamiento, temp_min, temp_media, temp_max FROM {TABLE_DEST} WHERE flag_revision = 1",
        cnxn
    )
    print(df_flags.to_string(index=False))

## 7 · Cerrar conexión

In [ ]:
cursor.close()
cnxn.close()
print("🔒 Conexión cerrada correctamente")